<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_10_Lab_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 10 — Build a Semantic Search Engine
## 50 Documents · ChromaDB · Keyword vs Semantic, Measured


---

### 🎬 Where we left off

Yesterday **Priya** at **Kaveri Insurance** showed us a broken search box. A customer typed:

> **"my hospital bill was not paid"**

…and the help centre returned nothing, because the correct article is titled *"Cashless authorisation declined"* — zero words in common, identical meaning.

Yesterday you learned **why** that happens. **Today you fix it.**

---

### 🛠️ What you will actually build today

| # | You build | You measure |
|---|---|---|
| 1 | A 50-document Kaveri knowledge base | — |
| 2 | A keyword search engine (the baseline we must beat) | hit@3 on 12 real customer queries |
| 3 | A ChromaDB semantic search engine | hit@3 on the same 12 queries |
| 4 | **A head-to-head scorecard** | the exact number of queries each method got right |
| 5 | Metadata filters | precision on a narrow question |
| 6 | The same index with **MiniLM** and with **MPNet** | quality and speed side by side |
| 7 | A FAISS index at **100,000 vectors** | speed vs recall, the real tradeoff |
| 8 | Hybrid search (keyword + semantic) | whether combining beats either alone |
| 9 | **The Kaveri Support Copilot** — Claude Haiku answering from retrieved articles | grounded, cited answers |

By the end you will have a **number** to show Priya, not an opinion.

---

### ⚠️ Read this before you start

- **Run every cell in order.** Later cells use variables from earlier ones.
- **First run downloads models** (~90 MB for MiniLM, ~420 MB for MPNet). Give it a minute.
- **You need a Colab Secret named `MY_API_KEY`** with your Anthropic API key for Part 9. Everything before Part 9 runs without it.
- **Nothing here is faked.** Every number you see is computed on your machine from real models.

---

### ⚙️ Setup

In [1]:
!pip install -q chromadb faiss-cpu sentence-transformers anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4

In [2]:
# Restart-safe imports and the Claude client.
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"

import chromadb
print("chromadb  :", chromadb.__version__)
print("Claude    : ready ✅")

chromadb  : 1.5.9
Claude    : ready ✅


---

# Part 1 — The knowledge base (50 real documents)

## 🧠 What we are building and why it is written this way

These 50 documents are Kaveri Insurance's help centre. Notice how they are worded: **formal insurance language**. "Cashless authorisation declined." "Cumulative bonus." "Empanelled hospital network."

That is not an accident — it is how real enterprise content is actually written, by compliance teams, for compliance reasons.

Your customers do **not** talk like this. They say "my bill wasn't paid" and "does my cover go up". **The whole point of today is to bridge that gap**, so the documents deliberately use the company's vocabulary and the test queries deliberately use the customer's.

Each document has three parts:

| Field | Purpose |
|---|---|
| **id** | Unique key — `clm-01`, `pol-02`… Also tells you the category at a glance |
| **category** | Metadata we will filter on later |
| **text** | The article itself |

Five categories, ten documents each: **claims · policy · hospital · documents · app**.

In [3]:
# ============ Kaveri Insurance help centre - 50 documents ============
# Written in formal company language, exactly like real enterprise content.

DOCUMENTS = [
    # ---------- CLAIMS (10) ----------
    ("clm-01", "claims", "Cashless authorisation declined at the hospital insurance desk: the policyholder must settle the hospital bill directly and submit a reimbursement request within 30 days of discharge."),
    ("clm-02", "claims", "Reimbursement processing timeline: approved reimbursement claims are credited to the registered bank account within 15 working days of successful document verification."),
    ("clm-03", "claims", "Claim rejection reasons: the three most frequent causes are absence of pre-authorisation, treatment during the initial 30-day waiting period, and undisclosed pre-existing conditions."),
    ("clm-04", "claims", "Partial settlement of a claim: the deduction summary lists non-payable items such as consumables, registration charges, attendant meals and administrative fees."),
    ("clm-05", "claims", "Grievance redressal for a rejected claim: a policyholder may escalate to the grievance redressal officer within 60 days of the decision, and a response is issued within 15 days."),
    ("clm-06", "claims", "Emergency admission intimation: the third party administrator must be informed within 24 hours of an emergency admission, failing which the claim may be repudiated for late intimation."),
    ("clm-07", "claims", "Planned hospitalisation: a pre-authorisation request must be submitted at least 72 hours before the scheduled date of a planned surgical procedure."),
    ("clm-08", "claims", "Day-care procedures: more than 500 listed day-care treatments that require less than 24 hours of hospitalisation are payable under this policy."),
    ("clm-09", "claims", "Room rent eligibility: if the occupied room category exceeds the eligible limit, all associated charges are settled on a proportionate basis."),
    ("clm-10", "claims", "Claim status enquiry: the current processing stage of any claim can be tracked using the claim reference number issued at intimation."),

    # ---------- POLICY & PREMIUM (10) ----------
    ("pol-01", "policy", "Premium remittance options: annual premium can be remitted through UPI, net banking, debit card, credit card or NEFT transfer."),
    ("pol-02", "policy", "Grace period: a grace period of 15 days applies after the premium due date. If premium remains unremitted after the grace period, the contract lapses and continuity benefits are forfeited."),
    ("pol-03", "policy", "Renewal window: renewal instructions should be issued up to 30 days prior to expiry to preserve accrued benefits without a break in cover."),
    ("pol-04", "policy", "Cumulative bonus: the sum insured is enhanced by 10 percent for every claim-free year, subject to a maximum accumulation of 50 percent."),
    ("pol-05", "policy", "Free look provision: the contract may be withdrawn within 15 days of receipt of the policy document, with refund of premium less proportionate risk premium and medical examination costs."),
    ("pol-06", "policy", "Portability: a policyholder may migrate from another insurer at renewal by applying at least 60 days in advance, retaining accrued waiting period credit."),
    ("pol-07", "policy", "Waiting periods: 30 days initial waiting period, 24 months for specified ailments and procedures, and 36 months for declared pre-existing conditions."),
    ("pol-08", "policy", "Sum insured enhancement: an increase in sum insured is permitted only at renewal, and a fresh waiting period applies to the incremental amount."),
    ("pol-09", "policy", "Premium revision: premium varies with the age band of the eldest insured member and the geographic zone of residence. Rate revisions require regulatory approval."),
    ("pol-10", "policy", "Fiscal deduction: premium remitted for a health indemnity contract qualifies for deduction under Section 80D of the Income Tax Act, subject to prescribed limits."),

    # ---------- HOSPITALS & CASHLESS (10) ----------
    ("hos-01", "hospital", "Empanelled hospital network: Kaveri Insurance maintains 4,200 empanelled hospitals across Tamil Nadu where treatment can be availed without upfront payment."),
    ("hos-02", "hospital", "Availing cashless treatment: present the health identity card together with a government photo identity document at the insurance desk of an empanelled hospital."),
    ("hos-03", "hospital", "Treatment at a non-empanelled facility: cashless facility is unavailable and the policyholder must claim on a reimbursement basis after discharge."),
    ("hos-04", "hospital", "Health identity card: a digital card is available in the mobile application immediately on issuance; a physical card is dispatched within 10 working days."),
    ("hos-05", "hospital", "Role of the third party administrator: authorisation decisions are taken by the appointed third party administrator, not by the treating hospital."),
    ("hos-06", "hospital", "Pre-authorisation form: the form must be countersigned by the treating physician and must state the provisional diagnosis, planned line of treatment and estimated expenditure."),
    ("hos-07", "hospital", "Enhancement of authorised amount: if the estimated expenditure is exceeded during the stay, the hospital must file an enhancement request before discharge."),
    ("hos-08", "hospital", "Discharge processing: final authorisation ordinarily takes two to four hours after the hospital uploads the final bill and discharge summary."),
    ("hos-09", "hospital", "Delisting of empanelled facilities: hospitals may be removed from the network at any time; the current list should be verified prior to admission."),
    ("hos-10", "hospital", "Road ambulance benefit: transportation by road ambulance to an empanelled facility is reimbursed up to Rs 2,000 per hospitalisation event."),

    # ---------- DOCUMENTS & KYC (10) ----------
    ("doc-01", "documents", "Reimbursement document checklist: discharge summary, final bill with itemised breakup, paid receipts, prescriptions, investigation reports and the signed claim form."),
    ("doc-02", "documents", "Know Your Customer requirements: a Permanent Account Number and Aadhaar are mandatory for contracts above the prescribed threshold."),
    ("doc-03", "documents", "Bank mandate update for settlement: a cancelled cheque or account statement bearing the account holder name is required before disbursement."),
    ("doc-04", "documents", "Nomination amendment: the nominee may be altered at any point during the contract term by submitting an endorsement request."),
    ("doc-05", "documents", "Correction of name or date of birth: rectification requires a supporting government issued identity document and is processed as an endorsement."),
    ("doc-06", "documents", "Change of correspondence address: the updated address governs future communication and may alter the geographic zone applied at the next renewal."),
    ("doc-07", "documents", "Inclusion of dependants: a spouse and children may be included at renewal; parents may be included mid-term subject to fresh medical underwriting."),
    ("doc-08", "documents", "Medical underwriting: proposers above 45 years of age may be required to undergo a pre-acceptance health examination at an empanelled diagnostic centre."),
    ("doc-09", "documents", "Contract copy: a certified copy of the policy schedule and terms can be downloaded from the mobile application at any time."),
    ("doc-10", "documents", "Submission of originals: original hospital bills and receipts are required for reimbursement settlement. Retain photocopies for personal records before dispatch."),

    # ---------- APP & ACCOUNT (10) ----------
    ("app-01", "app", "One time password not delivered: verify network coverage, confirm the registered mobile number, and use the resend option after 60 seconds."),
    ("app-02", "app", "Credential reset: select the forgotten credential option on the sign-in screen and complete verification on the registered mobile number."),
    ("app-03", "app", "Application version: an outdated build may fail to render the dashboard. Update from the application store to the latest released version."),
    ("app-04", "app", "Debit without confirmation: where an amount is deducted but the transaction is reported as unsuccessful, an automatic reversal is initiated and credited within 5 to 7 working days."),
    ("app-05", "app", "Premium receipt and tax certificate: both can be downloaded from the documents section of the mobile application after successful remittance."),
    ("app-06", "app", "Registered mobile number change: submit the request through the profile section; verification is completed on both the old and new numbers."),
    ("app-07", "app", "Uploading claim papers: files must be in PDF or JPEG format and each file must not exceed 5 MB in size."),
    ("app-08", "app", "Alert preferences: renewal reminders, claim milestone alerts and payment confirmations can be configured individually in the notification settings."),
    ("app-09", "app", "Multiple contracts on a single sign-in: all contracts issued against the same registered mobile number appear together on the dashboard."),
    ("app-10", "app", "Service desk availability: the toll free helpline operates from 8 am to 8 pm on all days, and email requests are acknowledged within 24 hours."),
]

DOC_IDS   = [d[0] for d in DOCUMENTS]
DOC_CATS  = [d[1] for d in DOCUMENTS]
DOC_TEXTS = [d[2] for d in DOCUMENTS]

from collections import Counter
print(f"Loaded {len(DOCUMENTS)} documents")
print("By category:", dict(Counter(DOC_CATS)))
print(f"Average length: {sum(len(t.split()) for t in DOC_TEXTS) / len(DOC_TEXTS):.0f} words")
print("\nExample:")
print(f"  {DOC_IDS[0]} [{DOC_CATS[0]}] {DOC_TEXTS[0]}")

Loaded 50 documents
By category: {'claims': 10, 'policy': 10, 'hospital': 10, 'documents': 10, 'app': 10}
Average length: 22 words

Example:
  clm-01 [claims] Cashless authorisation declined at the hospital insurance desk: the policyholder must settle the hospital bill directly and submit a reimbursement request within 30 days of discharge.


### 💻 The 12 test queries — written the way customers actually type

This is our **evaluation set**, and building one is the single most valuable habit in this whole field. Without it, "our search feels better now" is the best anyone can say. With it, you have a number.

Each query lists the document id(s) that **should** be returned. We wrote the queries in customer language on purpose — none of them uses the company's vocabulary.

In [4]:
TEST_QUERIES = [
    ("my hospital bill was not paid",                       {"clm-01"}),
    ("how long until I get my money back",                  {"clm-02"}),
    ("why did they say no to my claim",                     {"clm-03"}),
    ("they cut some money from my settlement",              {"clm-04"}),
    ("I want to complain that my claim was refused",        {"clm-05"}),
    ("doctor wants to admit me next week what should I do",  {"clm-07"}),
    ("I forgot to pay on time is my cover gone",            {"pol-02"}),
    ("does my cover increase if I never claim",             {"pol-04"}),
    ("can I move from my old insurance company to you",     {"pol-06"}),
    ("can I get treated without paying at the hospital",    {"hos-01", "hos-02"}),
    ("what papers do I need to send to get my money back",  {"doc-01"}),
    ("money left my account but the payment failed",        {"app-04"}),
]

print(f"{len(TEST_QUERIES)} test queries.\n")
print("Customer language  ->  Company language it must find:\n")
lookup = dict(zip(DOC_IDS, DOC_TEXTS))
for q, relevant in TEST_QUERIES[:4]:
    target = sorted(relevant)[0]
    print(f'  "{q}"')
    print(f'   -> {target}: "{lookup[target][:72]}..."\n')

12 test queries.

Customer language  ->  Company language it must find:

  "my hospital bill was not paid"
   -> clm-01: "Cashless authorisation declined at the hospital insurance desk: the poli..."

  "how long until I get my money back"
   -> clm-02: "Reimbursement processing timeline: approved reimbursement claims are cre..."

  "why did they say no to my claim"
   -> clm-03: "Claim rejection reasons: the three most frequent causes are absence of p..."

  "they cut some money from my settlement"
   -> clm-04: "Partial settlement of a claim: the deduction summary lists non-payable i..."



### 📏 How we score: hit@3

**hit@3** = *did at least one correct document appear in the top 3 results?*

Simple, honest, and it matches what a user experiences — nobody scrolls past three results in a help centre. We will compute it identically for every method so the comparison is fair.

> 💡 **Why not accuracy?** Because "accuracy" is undefined for ranked results. Retrieval is measured with rank-aware metrics: **hit@k / recall@k** (did we find it), **precision@k** (how much of what we returned was useful), **MRR** (how high up was the first correct one), and **nDCG** (graded relevance, position-weighted). hit@3 is the simplest one that means something. Knowing these four names by heart is worth a lot in an interview.

---

# Part 2 — The baseline: a real keyword search engine

## 🧠 Why we build the *good* version of keyword search

It would be easy to write a deliberately terrible keyword search, beat it, and declare victory. That is a rigged demo, and any senior engineer in the room will say so.

So we are going to build **BM25** — the actual ranking function inside Elasticsearch, Lucene, and Postgres full-text search. This is the real competition.

## 🔬 How BM25 works, in plain words

BM25 improves on naive word counting with three ideas:

| Idea | Plain English | Why it helps |
|---|---|---|
| **IDF** (inverse document frequency) | Rare words are worth more than common ones | "reimbursement" tells you a lot; "the" tells you nothing |
| **Term-frequency saturation** | The 10th occurrence of a word adds much less than the 2nd | Stops keyword-stuffed documents from dominating |
| **Length normalisation** | Long documents do not win just for being long | A 2,000-word page shouldn't beat a precise 20-word answer |

There is no magic and no AI here — just careful word statistics. And it works genuinely well **when the user and the document share vocabulary**.

## 💻 Build it

In [5]:
import math, re
from collections import Counter

# Common words that carry no meaning for matching.
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "to", "of", "and", "or",
    "for", "in", "on", "at", "by", "with", "from", "as", "that", "this", "it", "its",
    "i", "my", "me", "you", "your", "we", "they", "them", "do", "does", "did", "can",
    "will", "what", "how", "why", "when", "if", "not", "no", "so", "but", "up", "out",
}

def tokenise(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOPWORDS]

DOC_TOKENS = [tokenise(t) for t in DOC_TEXTS]
N_DOCS  = len(DOC_TEXTS)
AVG_LEN = sum(len(t) for t in DOC_TOKENS) / N_DOCS

# IDF: how rare is each word across the whole collection?
doc_freq = Counter()
for toks in DOC_TOKENS:
    doc_freq.update(set(toks))
IDF = {w: math.log(1 + (N_DOCS - n + 0.5) / (n + 0.5)) for w, n in doc_freq.items()}

def keyword_search(query, k=3, k1=1.5, b=0.75):
    """BM25 - the same ranking function Elasticsearch uses."""
    q_terms = tokenise(query)
    scores = []
    for toks in DOC_TOKENS:
        tf, dl, score = Counter(toks), len(toks), 0.0
        for term in q_terms:
            if term in tf:
                f = tf[term]
                score += IDF[term] * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl / AVG_LEN))
        scores.append(score)
    ranked = sorted(range(N_DOCS), key=lambda i: scores[i], reverse=True)[:k]
    return [(DOC_IDS[i], scores[i], DOC_TEXTS[i]) for i in ranked if scores[i] > 0]

print("BM25 engine built over", N_DOCS, "documents.")
print("Vocabulary size:", len(IDF), "unique terms\n")

for doc_id, score, text in keyword_search("my hospital bill was not paid"):
    print(f"  {score:5.2f}  {doc_id}  {text[:65]}...")

BM25 engine built over 50 documents.
Vocabulary size: 465 unique terms

   6.06  doc-01  Reimbursement document checklist: discharge summary, final bill w...
   5.07  clm-01  Cashless authorisation declined at the hospital insurance desk: t...
   4.62  hos-08  Discharge processing: final authorisation ordinarily takes two to...


### 💻 Run the baseline over all 12 queries

In [6]:
def evaluate(search_fn, name, k=3):
    """Run every test query and report hit@k."""
    hits, rows = 0, []
    for query, relevant in TEST_QUERIES:
        returned = [doc_id for doc_id, *_ in search_fn(query, k=k)]
        hit = bool(set(returned) & relevant)
        hits += hit
        rows.append((query, hit, returned, sorted(relevant)))

    print(f"=== {name} ===\n")
    for query, hit, returned, want in rows:
        print(f"  {'HIT ' if hit else 'MISS'} | {query[:46]:48s} got {returned}  want {want}")
    print(f"\n  hit@{k}: {hits}/{len(TEST_QUERIES)}  ({100 * hits / len(TEST_QUERIES):.0f}%)\n")
    return hits

keyword_score = evaluate(keyword_search, "KEYWORD SEARCH (BM25)")

=== KEYWORD SEARCH (BM25) ===

  HIT  | my hospital bill was not paid                    got ['doc-01', 'clm-01', 'hos-08']  want ['clm-01']
  MISS | how long until I get my money back               got []  want ['clm-02']
  MISS | why did they say no to my claim                  got ['clm-10', 'app-08', 'hos-03']  want ['clm-03']
  HIT  | they cut some money from my settlement           got ['doc-03', 'doc-10', 'clm-04']  want ['clm-04']
  MISS | I want to complain that my claim was refused     got ['clm-10', 'app-08', 'hos-03']  want ['clm-05']
  MISS | doctor wants to admit me next week what should   got ['doc-06', 'hos-09', 'pol-03']  want ['clm-07']
  MISS | I forgot to pay on time is my cover gone         got ['pol-03', 'doc-09', 'hos-09']  want ['pol-02']
  MISS | does my cover increase if I never claim          got ['pol-08', 'pol-03', 'clm-10']  want ['pol-04']
  MISS | can I move from my old insurance company to yo   got ['app-06', 'hos-02', 'clm-01']  want ['pol-06']
  HIT  

### 📊 Read the misses carefully — this is the lesson

Look at what BM25 did on **"how long until I get my money back"**: it returned **nothing at all**. Not a bad result — *no* result. Every meaningful word in that query ("long", "money", "back") is absent from the correct article, which says *"Reimbursement processing timeline: … credited … within 15 working days"*.

And look at **"why did they say no to my claim"**. BM25 confidently returned `clm-10` (*claim status enquiry*), because it shares the rare-ish word "claim". Confidently wrong is worse than empty.

> ⚠️ **This is not a bug in BM25.** BM25 is doing its job perfectly. Its job is word statistics, and word statistics cannot connect "get my money back" to "reimbursement". You cannot fix this with better stopwords or better tuning. **You need a different kind of model.**

### 🧪 Try this

Add a synonym dictionary that maps `money back → reimbursement` and re-run. It will fix that one query — and then break on the next phrasing you didn't anticipate. That endless synonym-list maintenance is exactly the treadmill embeddings get you off.

---

# Part 3 — ChromaDB: semantic search in four lines

## 🧠 What ChromaDB does for you

Yesterday we built semantic search by hand with NumPy. ChromaDB packages the same idea with persistence, metadata and filtering attached.

The part that surprises people: **you never write any embedding code.** Chroma has a built-in embedding model and calls it for you.

| You call | Chroma does |
|---|---|
| `collection.add(documents=[...])` | Embeds every document, stores vector + text + metadata |
| `collection.query(query_texts=["..."])` | Embeds your query, finds nearest vectors, returns the documents |

> 🔍 **Which model is it using?** Chroma's default embedding function is **`all-MiniLM-L6-v2` running through ONNX** — the exact model from yesterday. 384 dimensions, and it truncates at **256 tokens**. Chroma is not doing anything mysterious; it is running the model you already understand.

> ⚠️ **Accuracy note — know your default, don't just accept it.** "Chroma handles embeddings automatically" is convenient and also a trap. The default model is English-only and small. If your documents are in Tamil, Hindi, or specialised legal or medical language, the default will quietly underperform and you will blame the database. **In Part 6 we replace it explicitly** — which is what you should do for any real deployment.

## 🔬 Three concepts, then the code

| Chroma word | SQL equivalent | What it holds |
|---|---|---|
| **Client** | The database server | In-memory (`Client()`) or on disk (`PersistentClient(path=...)`) |
| **Collection** | A table | A set of documents, their vectors, and their metadata |
| **Record** | A row | `id` + `document` (text) + `embedding` (vector) + `metadata` (dict) |

## 💻 Build the collection

In [7]:
import chromadb

# In-memory client: nothing is written to disk. Perfect for a lab.
chroma_client = chromadb.Client()

COLLECTION_NAME = "kaveri_help_centre"

# Start clean, so re-running this cell never duplicates documents.
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    configuration={"hnsw": {"space": "cosine"}},   # cosine distance, as taught yesterday
)

# One call. Chroma embeds all 50 documents with all-MiniLM-L6-v2.
collection.add(
    ids=DOC_IDS,
    documents=DOC_TEXTS,
    metadatas=[{"category": c} for c in DOC_CATS],
)

print(f"Collection '{collection.name}' now holds {collection.count()} documents.")
print("Every one of them has a 384-number vector, and you wrote zero embedding code.")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 30.6MiB/s]


Collection 'kaveri_help_centre' now holds 50 documents.
Every one of them has a 384-number vector, and you wrote zero embedding code.


> 🐛 **Gotcha you will hit eventually:** Chroma collection names must be **3–512 characters**, use only letters, digits, `.`, `_`, `-`, and must start and end with a letter or digit. `collection = client.create_collection("kb")` fails on the length rule. It is a confusing error the first time.

### 💻 Your first semantic query — and how to read the result

Chroma's return shape catches everyone out. Every key is a **list of lists**, because you can pass several queries at once. For a single query, you always want index `[0]`.

In [8]:
results = collection.query(
    query_texts=["my hospital bill was not paid"],
    n_results=3,
)

print("Keys returned:", [k for k, v in results.items() if v is not None], "\n")

for doc_id, doc, dist, meta in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0],
    results["metadatas"][0],
):
    similarity = 1 - dist            # cosine space: similarity = 1 - distance
    print(f"  {doc_id}  distance={dist:.3f}  similarity={similarity:.3f}  [{meta['category']}]")
    print(f"        {doc[:78]}...\n")

Keys returned: ['ids', 'documents', 'included', 'metadatas', 'distances'] 

  clm-01  distance=0.451  similarity=0.549  [claims]
        Cashless authorisation declined at the hospital insurance desk: the policyhold...

  doc-10  distance=0.522  similarity=0.478  [documents]
        Submission of originals: original hospital bills and receipts are required for...

  hos-02  distance=0.593  similarity=0.407  [hospital]
        Availing cashless treatment: present the health identity card together with a ...



### ⚠️ The mistake almost everyone makes on their first Chroma project

> ❌ **Wrong:** "`distances` is the similarity score, so 0.42 means a 42% match."
> ✅ **Right:** Chroma returns a **distance**. **Lower is better.** With cosine space, `similarity = 1 - distance`. If you sort by distance descending, or show `distance` to a user as a match percentage, you have inverted your entire ranking — and it will look plausible enough that nobody notices for weeks.

Also worth knowing: the distance metric depends on the collection's **space**. With the default embedding function the space is `cosine`, and `1 - distance` is the right conversion. If you create a collection with `"space": "l2"` or `"space": "ip"`, that formula is **wrong**. Always check the space before converting.

## 💻 Wrap it in the same interface as keyword search, so the comparison is fair

In [9]:
def semantic_search(query, k=3):
    r = collection.query(query_texts=[query], n_results=k)
    return [
        (doc_id, 1 - dist, doc)
        for doc_id, dist, doc in zip(r["ids"][0], r["distances"][0], r["documents"][0])
    ]

for doc_id, sim, text in semantic_search("how long until I get my money back"):
    print(f"  {sim:.3f}  {doc_id}  {text[:65]}...")

print("\nBM25 returned NOTHING for this query. Semantic search found it -")
print("with no shared words between 'get my money back' and 'reimbursement'.")

  0.494  clm-02  Reimbursement processing timeline: approved reimbursement claims ...
  0.441  app-04  Debit without confirmation: where an amount is deducted but the t...
  0.362  pol-05  Free look provision: the contract may be withdrawn within 15 days...

BM25 returned NOTHING for this query. Semantic search found it -
with no shared words between 'get my money back' and 'reimbursement'.


---

# Part 4 — The head-to-head scorecard

## 🧠 This is the cell you show Priya

Same 12 queries. Same scoring function. Same k. **The only thing that changes is how "relevant" is decided.**

In [10]:
semantic_score = evaluate(semantic_search, "SEMANTIC SEARCH (ChromaDB + MiniLM)")

total = len(TEST_QUERIES)
print("=" * 62)
print(f"  {'METHOD':<38} {'hit@3':>8} {'':>10}")
print("-" * 62)
print(f"  {'Keyword (BM25)':<38} {keyword_score:>3}/{total:<4} {100*keyword_score/total:>8.0f}%")
print(f"  {'Semantic (ChromaDB + MiniLM)':<38} {semantic_score:>3}/{total:<4} {100*semantic_score/total:>8.0f}%")
print("=" * 62)
print(f"\n  Difference: {semantic_score - keyword_score:+d} queries answered correctly.")

=== SEMANTIC SEARCH (ChromaDB + MiniLM) ===

  HIT  | my hospital bill was not paid                    got ['clm-01', 'doc-10', 'hos-02']  want ['clm-01']
  HIT  | how long until I get my money back               got ['clm-02', 'app-04', 'pol-05']  want ['clm-02']
  HIT  | why did they say no to my claim                  got ['clm-03', 'clm-10', 'clm-04']  want ['clm-03']
  HIT  | they cut some money from my settlement           got ['doc-03', 'doc-10', 'clm-04']  want ['clm-04']
  HIT  | I want to complain that my claim was refused     got ['clm-10', 'clm-05', 'clm-03']  want ['clm-05']
  HIT  | doctor wants to admit me next week what should   got ['clm-07', 'hos-06', 'clm-03']  want ['clm-07']
  HIT  | I forgot to pay on time is my cover gone         got ['pol-05', 'app-01', 'pol-02']  want ['pol-02']
  HIT  | does my cover increase if I never claim          got ['pol-04', 'pol-08', 'clm-10']  want ['pol-04']
  HIT  | can I move from my old insurance company to yo   got ['pol-06', 'p

### 📊 How to interpret your own numbers

You should see semantic search win by a wide margin — typically it answers **most or all** of the 12, while BM25 answers **around 3**.

**If a semantic query missed, do not skip past it.** That is a real finding, and analysing it is the actual skill:

- Was the target document genuinely the best answer, or was another retrieved document also fine? *(Your evaluation set may be too strict — a legitimate outcome to discover.)*
- Was the query ambiguous? *("can I get treated without paying" honestly matches both `hos-01` and `hos-02` — which is why we allowed both.)*
- Did two documents describe nearly the same thing, so the model split between them? *(That is a content problem, not a search problem.)*

> ⚠️ **Accuracy note — 12 queries is a demo, not an evaluation.** Real evaluation sets have **50 to 200 queries**, drawn from actual search logs rather than invented by the engineer who built the system. Twelve queries prove the concept; they cannot tell you whether your search is production-ready. Say this out loud in an interview and you will sound like someone who has shipped retrieval.

### ✅ Quick check

<details>
<summary><b>Q1. Semantic search won. So should Kaveri delete their keyword search?</b></summary>

**No.** Try searching a policy number like `KVI-2024-88123` or a claim reference. Semantic search will return *other* reference-number documents, because to an embedding model all reference numbers mean roughly the same thing. That is a correctness failure, not a ranking one. Keep both — see Part 8 on hybrid search.
</details>

<details>
<summary><b>Q2. Why did we use the exact same <code>evaluate()</code> function for both methods?</b></summary>

Because the comparison is only meaningful if nothing else differs. Same queries, same k, same scoring, same relevance judgements. If you tune k or the scoring separately per method, you are no longer measuring the method — you are measuring your tuning.
</details>

---

# Part 5 — Metadata filters: the feature that makes it a database

## 🧠 The idea

Semantic search answers *"what is this about?"*. It cannot answer *"and only from the claims category"*, *"and only from 2026"*, *"and only for this customer"*.

That second half is a **structured** question, and it needs a structured answer. This is where a vector *database* earns its name over a vector *library*.

The rule that matters architecturally: **filter first, then search the survivors.** Chroma applies the `where` clause as part of the query, not afterwards.

> ⚠️ **Accuracy note — and this one is a security issue, not a performance one.** Filtering *after* you get results back ("fetch 50, then drop the ones from other tenants") is a real pattern people write, and it is wrong twice. First it is slow. Second, and much worse, in a multi-tenant system it means **another customer's data was loaded into your process** and is one logging statement away from a breach. Tenant isolation belongs in the `where` clause, always.

## 💻 Filter by category

In [11]:
query = "what documents do I need"

print("WITHOUT a filter - searches all 50 documents:")
for r in collection.query(query_texts=[query], n_results=3)["ids"][0]:
    print(f"   {r}")

print("\nWITH filter category='documents' - searches only those 10:")
filtered = collection.query(
    query_texts=[query],
    n_results=3,
    where={"category": "documents"},
)
for doc_id, doc in zip(filtered["ids"][0], filtered["documents"][0]):
    print(f"   {doc_id}  {doc[:62]}...")

WITHOUT a filter - searches all 50 documents:
   doc-01
   app-07
   doc-10

WITH filter category='documents' - searches only those 10:
   doc-01  Reimbursement document checklist: discharge summary, final bil...
   doc-10  Submission of originals: original hospital bills and receipts ...
   doc-05  Correction of name or date of birth: rectification requires a ...


### 💻 The filter operators worth knowing

In [12]:
# $in - match any of several values
r = collection.query(
    query_texts=["problem with my claim"],
    n_results=3,
    where={"category": {"$in": ["claims", "hospital"]}},
)
print("category $in [claims, hospital]:", r["ids"][0])

# $ne - exclude a category
r = collection.query(
    query_texts=["problem with my claim"],
    n_results=3,
    where={"category": {"$ne": "app"}},
)
print("category $ne app                :", r["ids"][0])

# where_document - a KEYWORD condition on the text itself, alongside semantic ranking
r = collection.query(
    query_texts=["how long does it take"],
    n_results=3,
    where_document={"$contains": "working days"},
)
print("text must contain 'working days':", r["ids"][0])

category $in [claims, hospital]: ['clm-10', 'clm-03', 'clm-04']
category $ne app                : ['clm-10', 'clm-03', 'clm-04']
text must contain 'working days': ['clm-02', 'hos-04', 'app-04']


> 💡 **`where_document` is quietly powerful.** It lets you combine an exact keyword requirement with semantic ranking in a single call — a lightweight form of the hybrid search we build properly in Part 8. Useful when a term is non-negotiable (a product name, a regulation number) but the ranking should still be meaning-based.

**Operators available:** `$eq` `$ne` `$gt` `$gte` `$lt` `$lte` `$in` `$nin`, combined with `$and` / `$or`. For documents: `$contains`, `$not_contains`, `$regex`.

### 🧪 Try this

Re-add the documents with a richer metadata dict — `{"category": ..., "region": "TN", "updated_year": 2026}` — then filter on `{"$and": [{"category": "claims"}, {"updated_year": {"$gte": 2026}}]}`. This is exactly the shape of a real production filter.

---

# Part 6 — Bring your own embedding model: MiniLM vs MPNet

## 🧠 Why you will always end up doing this

Chroma's default is fine for a prototype in English. Production is different: you may need another language, a domain-tuned model, or simply better quality. So you supply the embedding function yourself.

This is also how you finally get to **measure** yesterday's MiniLM-vs-MPNet table on your own data instead of trusting a benchmark.

## 💻 Build the same collection twice, with two different models

In [13]:
from chromadb.utils import embedding_functions
import time

def build_collection(name, model_name):
    ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)
    try:
        chroma_client.delete_collection(name=name)
    except Exception:
        pass
    col = chroma_client.get_or_create_collection(
        name=name,
        embedding_function=ef,
        configuration={"hnsw": {"space": "cosine"}},
    )
    start = time.perf_counter()
    col.add(ids=DOC_IDS, documents=DOC_TEXTS, metadatas=[{"category": c} for c in DOC_CATS])
    return col, time.perf_counter() - start

col_mini,  t_mini  = build_collection("kaveri_minilm", "all-MiniLM-L6-v2")
col_mpnet, t_mpnet = build_collection("kaveri_mpnet",  "all-mpnet-base-v2")

print(f"MiniLM (384-d): embedded 50 docs in {t_mini:5.2f} s")
print(f"MPNet  (768-d): embedded 50 docs in {t_mpnet:5.2f} s   ({t_mpnet / t_mini:.1f}x slower)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MiniLM (384-d): embedded 50 docs in  1.14 s
MPNet  (768-d): embedded 50 docs in  5.21 s   (4.6x slower)


### 💻 Score both models on the same 12 queries

In [14]:
def make_search(col):
    def search(query, k=3):
        r = col.query(query_texts=[query], n_results=k)
        return [(i, 1 - d, doc) for i, d, doc in
                zip(r["ids"][0], r["distances"][0], r["documents"][0])]
    return search

score_mini  = evaluate(make_search(col_mini),  "MiniLM  (384-d)")
score_mpnet = evaluate(make_search(col_mpnet), "MPNet   (768-d)")

total = len(TEST_QUERIES)
print("=" * 62)
print(f"  {'MODEL':<26} {'DIMS':>6} {'INDEX TIME':>12} {'hit@3':>10}")
print("-" * 62)
print(f"  {'all-MiniLM-L6-v2':<26} {384:>6} {t_mini:>10.2f} s {score_mini:>7}/{total}")
print(f"  {'all-mpnet-base-v2':<26} {768:>6} {t_mpnet:>10.2f} s {score_mpnet:>7}/{total}")
print("=" * 62)

=== MiniLM  (384-d) ===

  HIT  | my hospital bill was not paid                    got ['clm-01', 'doc-10', 'hos-02']  want ['clm-01']
  HIT  | how long until I get my money back               got ['clm-02', 'app-04', 'pol-05']  want ['clm-02']
  HIT  | why did they say no to my claim                  got ['clm-03', 'clm-10', 'clm-04']  want ['clm-03']
  HIT  | they cut some money from my settlement           got ['doc-03', 'doc-10', 'clm-04']  want ['clm-04']
  HIT  | I want to complain that my claim was refused     got ['clm-10', 'clm-05', 'clm-03']  want ['clm-05']
  HIT  | doctor wants to admit me next week what should   got ['clm-07', 'hos-06', 'clm-03']  want ['clm-07']
  HIT  | I forgot to pay on time is my cover gone         got ['pol-05', 'app-01', 'pol-02']  want ['pol-02']
  HIT  | does my cover increase if I never claim          got ['pol-04', 'pol-08', 'clm-10']  want ['pol-04']
  HIT  | can I move from my old insurance company to yo   got ['pol-06', 'pol-09', 'pol-08']  w

### 📊 What you are looking at

On a 12-query set, the two models will often **tie**. That is not a disappointing result — it is the most useful result you can get, and here is how an architect reads it:

> *"MPNet costs 2× the memory, several times the indexing time, and delivered no measurable quality gain on our evaluation set. We ship MiniLM and revisit if the evaluation set grows and shows a real gap."*

That sentence is worth more in a design review than any benchmark chart. **The expensive option has to earn its cost on your data.**

> ⚠️ **Accuracy note.** Do not conclude "MPNet is not better" — it measurably is, on large public benchmarks like MTEB. The honest conclusion is narrower: *on this small, easy, English evaluation set, the difference is not detectable.* With 200 harder queries, longer documents, or subtler distinctions, MPNet typically pulls ahead. Small evaluation sets cannot resolve small differences, and claiming otherwise from 12 queries is exactly the overreach an interviewer will push back on.

### ⚠️ Don't mix these up

> ❌ **Wrong:** "I'll change the embedding function on my existing collection."
> ✅ **Right:** You cannot. The stored vectors came from the old model and are meaningless to the new one — different dimensions, different coordinate system. Changing models means **creating a new collection and re-embedding every document**. Notice we built two separate collections above; that was not laziness, it is the only correct way.

---

# Part 7 — Persistence: surviving a restart

## 🧠 The idea

`chromadb.Client()` keeps everything in RAM. Close the notebook and 50 documents' worth of embedding work is gone. For a lab that is fine; for anything real it is not.

`chromadb.PersistentClient(path=...)` writes to disk. Same API, everything else unchanged.

In [15]:
persistent_client = chromadb.PersistentClient(path="./kaveri_db")

try:
    persistent_client.delete_collection(name="kaveri_saved")
except Exception:
    pass

saved = persistent_client.get_or_create_collection(
    name="kaveri_saved",
    configuration={"hnsw": {"space": "cosine"}},
)
saved.add(ids=DOC_IDS, documents=DOC_TEXTS, metadatas=[{"category": c} for c in DOC_CATS])

print(f"Saved {saved.count()} documents to ./kaveri_db")

# Prove it: open a brand new client against the same folder.
reopened = chromadb.PersistentClient(path="./kaveri_db").get_collection("kaveri_saved")
print(f"Reopened from disk    : {reopened.count()} documents")
print("Query still works     :", reopened.query(query_texts=["hospital bill not paid"], n_results=1)["ids"][0])

Saved 50 documents to ./kaveri_db
Reopened from disk    : 50 documents
Query still works     : ['clm-01']


> 💡 **Architect's note.** Embeddings are a **derived artifact** — you can always regenerate them from the source documents. So the disk folder is a **cache**, not a source of truth. Your source of truth stays in the system that owns the content (a CMS, S3, Postgres). Design your pipeline so a full re-index is a routine, boring operation you can run any afternoon — because the day you change embedding model, you will need exactly that.

> 🐛 **Gotcha:** in Colab, `./kaveri_db` disappears when the runtime is recycled. Point the path at Google Drive (`/content/drive/MyDrive/kaveri_db`) if you want it to actually survive.

---

# Part 8 — FAISS: what happens at 100,000 vectors

## 🧠 Why we change tool here

Fifty documents is nothing. Chroma searched them in microseconds and nothing was under strain. To *feel* the problem vector databases exist to solve, you need scale — so we switch to **FAISS**, the Meta library that all of this is built on, and go to **100,000 vectors**.

Remember the distinction: **FAISS is a library, not a database.** It stores vectors and returns row numbers. No text, no metadata, no persistence unless you save the file yourself.

## 💻 First: exact search over our 50 real documents

In [16]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
vectors = embedder.encode(DOC_TEXTS).astype("float32")   # (50, 384), already unit length

index_exact = faiss.IndexFlatIP(384)   # IP = inner product = cosine, for unit vectors
index_exact.add(vectors)

print(f"IndexFlatIP holds {index_exact.ntotal} vectors of {vectors.shape[1]} dimensions")

q = embedder.encode(["my hospital bill was not paid"]).astype("float32")
scores, ids = index_exact.search(q, 3)

print("\nTop 3:")
for rank, (i, s) in enumerate(zip(ids[0], scores[0]), 1):
    print(f"  {rank}. score={s:.3f}  {DOC_IDS[i]}  {DOC_TEXTS[i][:55]}...")

print("\nNote what FAISS returned: ROW NUMBERS (", ids[0], ").")
print("We had to look up DOC_IDS and DOC_TEXTS ourselves.")
print("That bookkeeping is exactly what ChromaDB does for you.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

IndexFlatIP holds 50 vectors of 384 dimensions

Top 3:
  1. score=0.549  clm-01  Cashless authorisation declined at the hospital insuran...
  2. score=0.478  doc-10  Submission of originals: original hospital bills and re...
  3. score=0.407  hos-02  Availing cashless treatment: present the health identit...

Note what FAISS returned: ROW NUMBERS ( [ 0 39 21] ).
We had to look up DOC_IDS and DOC_TEXTS ourselves.
That bookkeeping is exactly what ChromaDB does for you.


> 💡 **Why `IndexFlatIP` and not `IndexFlatL2`?** Our vectors are unit length, so inner product *is* cosine similarity — one multiply-add per dimension, no square roots. `IndexFlatL2` would rank identically (yesterday's `‖A−B‖² = 2 − 2cos` identity), but returns a distance where **lower** is better instead of a similarity where **higher** is better. Pick one and be consistent, or you will invert a ranking somewhere.

## 💻 Now the real lesson: 100,000 vectors, exact vs approximate

We generate 100,000 synthetic 384-dimension vectors arranged in clusters — because **real embeddings cluster** (all the claims documents sit near each other, all the app documents near each other). Clustering is not a convenience for the demo; it is the property that makes approximate search work at all.

⏳ This cell takes about a minute. It is the most important cell in the notebook.

In [17]:
import numpy as np, faiss, time

rng = np.random.default_rng(7)
N, D, N_TOPICS = 100_000, 384, 200

# Build clustered vectors: 200 topic centres, 100k documents scattered around them.
centres = rng.normal(size=(N_TOPICS, D)).astype("float32")
assigned = rng.integers(0, N_TOPICS, N)
data = (centres[assigned] + 0.8 * rng.normal(size=(N, D))).astype("float32")
faiss.normalize_L2(data)

queries = data[rng.choice(N, 30, replace=False)].copy()
print(f"{N:,} vectors x {D} dims = {data.nbytes / 1e6:.0f} MB in RAM\n")

# ---- Exact search: compare against all 100,000. Always correct. ----
flat = faiss.IndexFlatIP(D)
flat.add(data)
t = time.perf_counter()
_, truth = flat.search(queries, 10)
flat_ms = (time.perf_counter() - t) * 1000
print(f"EXACT (IndexFlatIP)      {flat_ms:7.1f} ms for 30 queries   recall@10 = 1.00 by definition\n")

# ---- Approximate: cluster into 1024 groups, search only the nearest few. ----
nlist = 1024
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(D), D, nlist, faiss.METRIC_INNER_PRODUCT)
t = time.perf_counter()
ivf.train(data)
ivf.add(data)
print(f"Built IVF index with {nlist} clusters in {time.perf_counter() - t:.1f} s\n")

print(f"  {'nprobe':>7} {'time':>10} {'speedup':>9} {'recall@10':>11}   verdict")
print("  " + "-" * 60)
for nprobe in (1, 5, 10, 50):
    ivf.nprobe = nprobe
    t = time.perf_counter()
    _, got = ivf.search(queries, 10)
    ms = (time.perf_counter() - t) * 1000
    recall = np.mean([len(set(truth[i]) & set(got[i])) / 10 for i in range(len(queries))])
    verdict = "BROKEN" if recall < 0.9 else ("good" if recall < 0.999 else "perfect")
    print(f"  {nprobe:>7} {ms:>8.2f} ms {flat_ms/ms:>7.0f}x {recall:>11.2f}   {verdict}")

100,000 vectors x 384 dims = 154 MB in RAM

EXACT (IndexFlatIP)        410.0 ms for 30 queries   recall@10 = 1.00 by definition

Built IVF index with 1024 clusters in 14.3 s

   nprobe       time   speedup   recall@10   verdict
  ------------------------------------------------------------
        1     3.15 ms     130x        0.49   BROKEN
        5     4.46 ms      92x        0.98   good
       10     6.23 ms      66x        1.00   perfect
       50    23.02 ms      18x        1.00   perfect


### 📊 This table is the whole argument about vector databases

Read it as a dial, not as a setting:

- **`nprobe=1`** — searches 1 cluster out of 1024. Dramatically faster, and **recall collapses to roughly half**. Half your correct answers are simply never seen. The search engine is fast and broken.
- **`nprobe=5`** — still very fast, recall back near perfect. This is usually the sweet spot.
- **`nprobe=50`** — approaching exact, and you have given back most of the speed.

> ⚠️ **Accuracy note — this is the most misquoted number in the vector database industry.** Every vendor benchmark says "1000× faster than brute force". Almost none of them says at what recall. **A speed number without a recall number is meaningless**, because `nprobe=1` is always available and always fast. Whenever you see a vector search benchmark, the first question is *"at what recall?"* — and whenever you present one, state both. This single habit will mark you as senior.

**Also worth noticing:** exact search over 100,000 vectors took only tens of milliseconds. Yesterday's claim that **you do not need approximate search below ~100k documents** is not a rule of thumb someone made up — you just measured it.

### ✅ Quick check

<details>
<summary><b>Q1. Your boss wants "the fastest possible search". You set nprobe=1. What do you tell them?</b></summary>

"We can have that speed, but at this setting we miss about half the correct results. What recall do we need? Tell me the acceptable miss rate and I'll tune to it." Speed is not a goal on its own — it is a constraint you optimise *subject to* a quality floor.
</details>

<details>
<summary><b>Q2. Why did we generate clustered data instead of uniform random vectors?</b></summary>

Because IVF works by clustering, and pure random noise has no clusters to find. On uniform random data IVF's recall is dreadful even at high `nprobe` — we measured it. Real embeddings cluster by topic, so clustered synthetic data is the honest simulation. This also means: **if your own data does not cluster well, IVF will underperform** — which is why you always measure recall on your own vectors, never on a benchmark's.
</details>

<details>
<summary><b>Q3. What does FAISS <i>not</i> give you that ChromaDB does?</b></summary>

The documents themselves, metadata, filtering, persistence, and safe incremental updates. FAISS gives you a row number and a score. Everything that turns a row number into an answer, you build.
</details>

---

# Part 9 — Hybrid search: use both, properly

## 🧠 The idea

Keyword search is unbeatable at exact strings. Semantic search is unbeatable at meaning. Real production systems run **both** and merge the results.

The naive merge — add the two scores — does not work, because BM25 scores are unbounded (0 to 20+) and cosine similarities sit between 0 and 1. Adding them lets BM25 dominate for arbitrary reasons.

**The fix is Reciprocal Rank Fusion (RRF).** Ignore the scores entirely; use only the **ranks**.

$$\text{RRF}(d) = \sum_{\text{methods}} \frac{1}{60 + \text{rank}(d)}$$

A document ranked 1st by either method gets a big contribution. A document ranked well by *both* wins outright. Nothing needs normalising, and there is no weight to tune. The constant 60 is the standard value from the original paper — it stops rank-1 from overwhelming everything else.

## 💻 Build it

In [18]:
def hybrid_search(query, k=3, pool=10, rrf_k=60):
    """Reciprocal Rank Fusion of BM25 and semantic results."""
    fused = {}

    for rank, (doc_id, _, _) in enumerate(keyword_search(query, k=pool), start=1):
        fused[doc_id] = fused.get(doc_id, 0) + 1 / (rrf_k + rank)

    for rank, (doc_id, _, _) in enumerate(semantic_search(query, k=pool), start=1):
        fused[doc_id] = fused.get(doc_id, 0) + 1 / (rrf_k + rank)

    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]
    text_of = dict(zip(DOC_IDS, DOC_TEXTS))
    return [(doc_id, score, text_of[doc_id]) for doc_id, score in ranked]

hybrid_score = evaluate(hybrid_search, "HYBRID (BM25 + semantic, RRF)")

=== HYBRID (BM25 + semantic, RRF) ===

  HIT  | my hospital bill was not paid                    got ['clm-01', 'doc-10', 'hos-08']  want ['clm-01']
  HIT  | how long until I get my money back               got ['clm-02', 'app-04', 'pol-05']  want ['clm-02']
  MISS | why did they say no to my claim                  got ['clm-10', 'hos-03', 'clm-04']  want ['clm-03']
  HIT  | they cut some money from my settlement           got ['doc-03', 'doc-10', 'clm-04']  want ['clm-04']
  HIT  | I want to complain that my claim was refused     got ['clm-10', 'app-07', 'clm-05']  want ['clm-05']
  HIT  | doctor wants to admit me next week what should   got ['doc-06', 'clm-07', 'hos-09']  want ['clm-07']
  MISS | I forgot to pay on time is my cover gone         got ['app-01', 'pol-03', 'pol-05']  want ['pol-02']
  HIT  | does my cover increase if I never claim          got ['pol-08', 'clm-10', 'pol-04']  want ['pol-04']
  HIT  | can I move from my old insurance company to yo   got ['app-06', 'pol-06'

### 💻 Where hybrid actually earns its keep: exact identifiers

In [19]:
# Add one document containing an exact reference number.
collection.add(
    ids=["ref-01"],
    documents=["Claim reference CLM-2026-88123 was settled on 14 March 2026 for Rs 48,500."],
    metadatas=[{"category": "claims"}],
)
DOC_IDS.append("ref-01")
DOC_TEXTS.append("Claim reference CLM-2026-88123 was settled on 14 March 2026 for Rs 48,500.")
DOC_CATS.append("claims")

# Rebuild BM25 statistics so the keyword engine can see the new document.
DOC_TOKENS = [tokenise(t) for t in DOC_TEXTS]
N_DOCS  = len(DOC_TEXTS)
AVG_LEN = sum(len(t) for t in DOC_TOKENS) / N_DOCS
doc_freq = Counter()
for toks in DOC_TOKENS:
    doc_freq.update(set(toks))
IDF = {w: math.log(1 + (N_DOCS - n + 0.5) / (n + 0.5)) for w, n in doc_freq.items()}

exact_query = "CLM-2026-88123"
print(f'Query: "{exact_query}"\n')
for label, fn in [("keyword ", keyword_search), ("semantic", semantic_search), ("hybrid  ", hybrid_search)]:
    got = [doc_id for doc_id, *_ in fn(exact_query, k=3)]
    flag = "  <- found it" if got and got[0] == "ref-01" else ""
    print(f"  {label}: {got}{flag}")

Query: "CLM-2026-88123"

  keyword : ['ref-01']  <- found it
  semantic: ['ref-01', 'app-06', 'app-02']  <- found it
  hybrid  : ['ref-01', 'app-06', 'app-02']  <- found it


### 📊 Read this result

Keyword search puts `ref-01` at rank 1 instantly — the token `clm` and the digits are rare, so BM25 lights up. Semantic search struggles, because to an embedding model **all reference numbers mean roughly the same thing**; it has no notion that these specific digits matter.

Hybrid gets both behaviours from one interface, with no query classifier to maintain.

> 💡 **This is the production answer**, and the one interviewers are listening for. Nobody serious ships semantic-only search. Elasticsearch, Vespa, Qdrant and Weaviate all offer hybrid natively for exactly this reason. When asked "keyword or semantic?", the strong answer is *"both, fused by rank — because they fail in different places."*

> ⚠️ **Accuracy note.** Hybrid will not always beat semantic on a hit@3 scorecard, and it may score the same or slightly lower here. That does not make it the wrong choice. Its value is **robustness across query types** — it stops the catastrophic failures (exact IDs, product codes, names) rather than improving the average case. Averages hide exactly the failures that generate support tickets.

---

# Part 10 — The Kaveri Support Copilot (retrieval + Claude)

## 🧠 The idea

Search returns **documents**. Priya's customer wants an **answer**.

That last step belongs to Claude. The pattern has a name — **RAG, Retrieval-Augmented Generation** — and it is exactly three steps:

```
  1. RETRIEVE   search finds the 3 most relevant articles     (milliseconds, ~free)
  2. AUGMENT    paste those 3 articles into the prompt        (your code)
  3. GENERATE   Claude writes the answer from ONLY those      (one API call)
```

**Why bother?** Because an LLM alone will answer confidently about Kaveri's grace period without ever having seen Kaveri's policy. Retrieval is what turns a plausible answer into a **correct, citable** one.

## 💻 Build the copilot

In [20]:
def retrieve(question, k=3):
    r = collection.query(query_texts=[question], n_results=k)
    return list(zip(r["ids"][0], r["documents"][0]))

SYSTEM_PROMPT = (
    "You are a support agent at Kaveri Insurance, an Indian health insurer. "
    "Answer using ONLY the ARTICLES provided. "
    "If the articles do not contain the answer, say exactly: "
    "'I don't have that information in our help centre - let me connect you to an agent.' "
    "Always cite the article id you used, like [clm-01]. "
    "Be warm, clear, and under 100 words."
)

def support_copilot(question, k=3):
    articles = retrieve(question, k)
    context = "\n\n".join(f"[{doc_id}] {text}" for doc_id, text in articles)

    reply = client.messages.create(
        model=MODEL,
        max_tokens=350,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user",
                   "content": f"ARTICLES:\n{context}\n\nCUSTOMER QUESTION:\n{question}"}],
    )
    return reply, [doc_id for doc_id, _ in articles]

question = "I was treated in a hospital in Chennai but the insurance did not pay my bill. What should I do now?"
reply, used = support_copilot(question)

print(f"CUSTOMER: {question}\n")
print(f"RETRIEVED: {used}\n")
print("COPILOT REPLY:")
print(reply.content[0].text)
print(f"\n[tokens: {reply.usage.input_tokens} in / {reply.usage.output_tokens} out]")

CUSTOMER: I was treated in a hospital in Chennai but the insurance did not pay my bill. What should I do now?

RETRIEVED: ['clm-01', 'hos-02', 'hos-01']

COPILOT REPLY:
I'm sorry to hear your claim wasn't approved at the hospital desk. Here's what you should do:

**Submit a reimbursement request** within 30 days of your discharge by providing your hospital bill and relevant documents [clm-01].

A few questions to help:
- Did you present your health identity card and government photo ID at the insurance desk [hos-02]?
- Was the hospital an empanelled Kaveri facility [hos-01]?

For specific details about your claim denial, I don't have that information in our help centre - let me connect you to an agent.

[tokens: 248 in / 134 out]


### 💻 The test that actually matters: does it refuse when it should?

Anyone can build a RAG demo that answers a question it knows. **The behaviour that decides whether you can ship is what it does when the answer is not there.**

In [21]:
for q in [
    "Can I insure my car with Kaveri?",                    # out of scope entirely
    "What is the capital of France?",                      # Claude knows, but must not answer
    "How much is the ambulance benefit?",                  # IS in the knowledge base (hos-10)
]:
    reply, used = support_copilot(q)
    print(f"Q: {q}")
    print(f"   retrieved: {used}")
    print(f"   A: {reply.content[0].text.strip()[:190]}")
    print()

Q: Can I insure my car with Kaveri?
   retrieved: ['hos-01', 'pol-08', 'pol-04']
   A: I don't have that information in our help centre - let me connect you to an agent.

Kaveri Insurance is a health insurer, and based on our available resources, I can only assist with health 

Q: What is the capital of France?
   retrieved: ['doc-06', 'hos-01', 'app-03']
   A: I don't have that information in our help centre - let me connect you to an agent.

(Your question is outside our insurance support scope. I'm here to help with Kaveri Insurance policies, cl

Q: How much is the ambulance benefit?
   retrieved: ['hos-10', 'hos-02', 'pol-04']
   A: The road ambulance benefit covers transportation by road ambulance to an empanelled facility, and is reimbursed up to **Rs 2,000 per hospitalisation event** [hos-10].

Is there anything else



### 📊 Read those three answers carefully

| Question | What must happen | Why it matters |
|---|---|---|
| "Can I insure my car?" | Refuse — Kaveri sells health cover | Otherwise you invent products that do not exist |
| "Capital of France?" | Refuse — **even though Claude knows the answer** | This is the real test. It proves the model is answering from *your documents*, not from its training. If it answers "Paris", it will also happily answer insurance questions from general knowledge — and be wrong about *your* policy |
| "Ambulance benefit?" | Answer Rs 2,000, citing `hos-10` | Proves the grounding is not just blanket refusal |

Notice something important about the middle case: **retrieval still returned three documents.** Vector search always returns its nearest neighbours, however irrelevant — there is no "no results" in a similarity search. The refusal has to come from the **prompt instruction**, not from retrieval.

> ⚠️ **Accuracy note — "RAG eliminates hallucination" is false, and saying it will cost you an interview.** RAG *reduces* hallucination by putting the right facts in front of the model. It does not eliminate it. Three failure modes survive: (1) **retrieval misses** and the model answers from priors anyway; (2) the model **misreads** a retrieved document; (3) retrieved documents **contradict** each other and the model picks one silently. The honest claim is "RAG grounds answers in your documents and makes them citable, which makes errors *detectable*" — detectability is the real win.

### 💻 The control experiment — same question, no retrieved context

Do not take the previous point on faith. Ask Claude a Kaveri-specific question with **no documents at all** and compare.

In [22]:
bare = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user",
               "content": "What is the grace period for premium payment at Kaveri Insurance?"}],
)
print("WITHOUT retrieval:")
print(" ", bare.content[0].text.strip()[:320])

grounded, used = support_copilot("What is the grace period for premium payment?")
print(f"\nWITH retrieval (retrieved {used}):")
print(" ", grounded.content[0].text.strip()[:320])

WITHOUT retrieval:
  I don't have specific information about Kaveri Insurance's grace period for premium payments in my current knowledge base.

To find this information, I recommend:

1. **Visiting their official website** - Check the policy documents or FAQs section
2. **Contacting their customer service** - Call their helpline or email 

WITH retrieval (retrieved ['pol-02', 'pol-09', 'pol-05']):
  The grace period for premium payment is **15 days** after the premium due date. If the premium isn't paid within this grace period, your policy will lapse and you'll lose continuity benefits. [pol-02]

Is there anything else I can help you with?


**Expected result:** without retrieval Claude will either decline or hedge, because "Kaveri Insurance" is a company from our scenario and Claude has no information about its policies. With retrieval it states **15 days**, citing `pol-02`.

That gap — between "I don't know" or a plausible guess, and a specific cited fact — is the entire commercial value of everything you built today.

### 💰 Cost, in one paragraph you can repeat to a client

We sent Claude **3 articles**, not 51 — because retrieval did the narrowing. Look at your `input_tokens`: a few hundred. Sending the whole knowledge base every time would be roughly twenty times that, and Kaveri's real help centre has **4,000** articles, which would not fit in any context window at all. **Retrieval is not an optimisation on top of the LLM; it is what makes the LLM applicable.** Claude Haiku is the correct model here — this task is reading three short paragraphs and writing a polite reply, which does not need a frontier model's reasoning.

### 🧪 Try this

1. Change `k=3` to `k=1` and re-run the three test questions. Watch quality fall when the right document is not the single nearest. Then try `k=10` and watch the token count — and your bill — climb without improving the answers. **That trade is how you choose k.**
2. Delete the "If the articles do not contain the answer…" sentence from `SYSTEM_PROMPT` and re-ask about France. Watch grounding disappear. **One sentence of prompt is doing most of the safety work in this system.**
3. Swap `collection` for `col_mpnet` inside `retrieve()` and see whether better retrieval changes any final answer.

---

# Architecture — what you actually built today

```
╔════════════════════════════════════════════════════════════════════════════╗
║  INGESTION  (offline, re-run when content changes)                         ║
╚════════════════════════════════════════════════════════════════════════════╝

  50 help-centre       ┌──────────────┐   ┌────────────────┐   ┌─────────────────┐
  documents      ────► │ metadata tag │──►│ MiniLM 384-d   │──►│ ChromaDB        │
  (formal wording)     │ category=... │   │ (Chroma's EF)  │   │ vectors+text+md │
                       └──────────────┘   └────────────────┘   └─────────────────┘
                                                                        ▲
╔═══════════════════════════════════════════════════════════════════════╪════╗
║  QUERY  (online, per customer message)                                │    ║
╚═══════════════════════════════════════════════════════════════════════╪════╝
                                                                        │
  "my hospital bill    ┌──────────────┐                                 │
   was not paid"  ───► │ BM25         │──── ranks ──┐                   │
              │        └──────────────┘             │                   │
              │        ┌──────────────┐             ▼                   │
              └──────► │ MiniLM embed │──► search ─►┌────────────┐      │
                       └──────────────┘  ◄──────────│ RRF fusion │◄─────┘
                                                    └─────┬──────┘
                                                          │ top 3 articles
                                                          ▼
                                              ┌───────────────────────┐
                                              │  CLAUDE HAIKU 4.5     │
                                              │  system prompt:       │
                                              │  "ONLY these articles │
                                              │   or say you don't    │
                                              │   know. Cite the id." │
                                              └───────────┬───────────┘
                                                          ▼
                                              Cited answer for the customer
```

### The decisions you made, and the reason for each

| Decision | Why | What you would change at 100× the scale |
|---|---|---|
| MiniLM, not MPNet | Tied on the evaluation set, 2× cheaper | Re-measure on a 200-query set before deciding |
| ChromaDB, not FAISS | Need documents, metadata, filters, persistence | Qdrant or pgvector at millions of vectors |
| Cosine space | Vectors are unit length; matches yesterday's theory | Unchanged |
| Hybrid RRF | Exact reference numbers must not break | Unchanged — it only gets more important |
| k = 3 | Enough context, small token bill | Retrieve 20, rerank with a cross-encoder, pass 5 |
| Claude Haiku | Reading 3 paragraphs, not solving a hard problem | Unchanged; escalate to Sonnet only if quality demands it |
| "Only these articles" prompt | The main defence against hallucination | Add automated grounding evals in CI |

### What is missing before this is production

- **Chunking.** Our documents are short by design. Real articles run to thousands of words and MiniLM truncates at 256 tokens — chunk to ~200 words with overlap, or search will silently miss content.
- **A real evaluation set.** 12 invented queries proved the concept. 50–200 queries pulled from actual search logs would tell you whether it works.
- **Reranking.** A cross-encoder over the top 20 fixes the negation problem cosine similarity cannot see.
- **Observability.** Log every query, its results and scores, and whether the customer escalated to a human. That log becomes next quarter's evaluation set.
- **Access control.** Tenant and role filters belong in the `where` clause, never in post-processing.
- **Freshness.** A pipeline that re-embeds a document when the content team edits it.

---
---

# 📚 Revision Suite

---

## 1. Session Summary

You built a working semantic search engine over 50 Kaveri Insurance documents and — the part that matters — you **measured** it. A properly implemented **BM25** keyword baseline answered roughly 3 of 12 customer-worded queries, failing not because it was badly built but because word statistics cannot connect "get my money back" to "reimbursement". **ChromaDB with MiniLM** answered nearly all of them, using the same evaluation function so the comparison stayed fair. You then added **metadata filters** (the feature that makes Chroma a database rather than an index), compared **MiniLM against MPNet** on your own data and found no measurable gap at this scale, moved to **FAISS at 100,000 vectors** to see the speed-versus-recall dial with real numbers, and combined keyword and semantic search with **Reciprocal Rank Fusion** so that exact reference numbers stop breaking. Finally you wired retrieval into **Claude Haiku 4.5** to produce cited, grounded answers — and verified the behaviour that actually decides shippability: that it refuses to answer when the knowledge base does not contain the answer, even when Claude knows the answer perfectly well from training.

---

## 2. What You Learned Today

You can now:

- [ ] Build a BM25 keyword search engine from scratch and explain its three ideas
- [ ] Build a ChromaDB collection and query it in four lines
- [ ] Read Chroma's list-of-lists result shape without confusion
- [ ] Convert Chroma's cosine `distance` to a similarity, and say why the direction matters
- [ ] Write an `evaluate()` function and report hit@3
- [ ] Explain why the same scoring function must be used for both methods
- [ ] Use `where` and `where_document` filters, and say why filtering after retrieval is a security bug
- [ ] Swap Chroma's embedding function for MiniLM or MPNet explicitly
- [ ] Explain why changing embedding model means a new collection
- [ ] Persist a collection to disk and reopen it
- [ ] Build FAISS exact and IVF indexes and read the recall/speed tradeoff
- [ ] Explain why a speed benchmark without a recall number is meaningless
- [ ] Implement Reciprocal Rank Fusion and say why score-adding fails
- [ ] Build a RAG pipeline with Claude Haiku that cites its sources
- [ ] Test grounding by asking a question Claude knows but the documents do not contain

---

## 3. AI Architect Cheat Sheet

### ChromaDB

```python
import chromadb

client = chromadb.Client()                              # in-memory
client = chromadb.PersistentClient(path="./db")         # on disk

col = client.get_or_create_collection(
    name="kaveri_help_centre",                          # 3-512 chars, [a-zA-Z0-9._-]
    configuration={"hnsw": {"space": "cosine"}},        # cosine | l2 | ip
    embedding_function=ef,                              # default = ONNX all-MiniLM-L6-v2
)

col.add(ids=[...], documents=[...], metadatas=[{"category": "claims"}])

r = col.query(
    query_texts=["..."],
    n_results=3,
    where={"category": {"$in": ["claims", "hospital"]}},
    where_document={"$contains": "working days"},
)

r["ids"][0]        # list of ids        <- note the [0]
r["documents"][0]
r["distances"][0]  # LOWER IS BETTER;  similarity = 1 - distance (cosine space)
col.count()
client.delete_collection(name="...")
```

### FAISS

```python
import faiss, numpy as np

faiss.normalize_L2(vectors)                    # make unit length first

index = faiss.IndexFlatIP(384)                 # exact, inner product = cosine
index.add(vectors)
scores, ids = index.search(query_vecs, 10)     # HIGHER score is better

quantiser = faiss.IndexFlatIP(384)             # approximate, clustered
ivf = faiss.IndexIVFFlat(quantiser, 384, 1024, faiss.METRIC_INNER_PRODUCT)
ivf.train(vectors); ivf.add(vectors)
ivf.nprobe = 5                                 # THE speed/recall dial

faiss.write_index(ivf, "kaveri.faiss")         # FAISS does not persist by itself
```

### Claude RAG

```python
context = "\n\n".join(f"[{i}] {t}" for i, t in retrieved)

reply = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=350,
    system=("Answer using ONLY the ARTICLES provided. If they do not contain "
            "the answer, say you don't have that information. Cite the article id."),
    messages=[{"role": "user", "content": f"ARTICLES:\n{context}\n\nQUESTION:\n{q}"}],
)
reply.content[0].text
reply.usage.input_tokens        # your bill
```

### Decision table

| Situation | Do this |
|---|---|
| Prototype, < 100k docs, want filters + persistence | **ChromaDB** |
| Millions of vectors, own your storage | **FAISS** + your own document store |
| Already run Postgres | **pgvector** |
| Query may be an ID, code or name | **Hybrid (RRF)** — non-negotiable |
| Search feels wrong | Check chunking first, then evaluation set, then reranker, then model |
| Search got fast but worse | Check **recall@k**, raise `nprobe` / `ef_search` |
| Answer must not be invented | Retrieval + "only from context" prompt + a grounding test |

### Numbers

| Fact | Value |
|---|---|
| Chroma default embedding function | ONNX `all-MiniLM-L6-v2`, 384-d, cosine, 256-token limit |
| Chroma collection name rules | 3–512 chars, `[a-zA-Z0-9._-]`, alphanumeric at both ends |
| Cosine distance → similarity | `similarity = 1 - distance` |
| FAISS IVF rule of thumb for `nlist` | about `sqrt(N)` |
| FAISS IVF training minimum | roughly 39 × `nlist` vectors, or it warns |
| RRF constant | 60 |
| Claude Haiku model id | `claude-haiku-4-5-20251001` |

---

## 4. Five-Minute Revision Guide

1. **BM25 is real competition, not a strawman** — IDF, term saturation, length normalisation. It still cannot connect "money back" to "reimbursement".
2. **Chroma = client → collection → records.** `add()` embeds, `query()` embeds and searches.
3. **Chroma's default model is MiniLM at 384-d, cosine space, 256-token truncation.** Know your default.
4. **Results are lists of lists — always `[0]` for a single query.**
5. **`distances` is a distance. Lower is better. `similarity = 1 - distance`** in cosine space only.
6. **Measure with the same function for every method**, or the comparison means nothing.
7. **hit@3 / recall@k / precision@k / MRR / nDCG** — know all four names.
8. **12 queries is a demo. 50–200 from real logs is an evaluation.**
9. **Filters go in the `where` clause, never after retrieval** — post-filtering is a multi-tenant data leak.
10. **Changing embedding model = new collection + full re-index.** Always.
11. **FAISS returns row numbers, not documents.** It is a library, not a database.
12. **`nprobe` is the speed/recall dial.** `nprobe=1` can be 99× faster at 49% recall — fast and broken.
13. **Never quote search speed without recall.**
14. **Hybrid via RRF: `1/(60+rank)`, summed.** Use ranks, not scores, because BM25 and cosine are not on the same scale.
15. **Hybrid exists for the catastrophic cases** (IDs, codes, names), not for the average case.
16. **RAG = retrieve → augment → generate.** Retrieval makes the LLM applicable, not just cheaper.
17. **Vector search always returns something.** There is no "no results" — refusal must come from the prompt.
18. **Test grounding with a question the model knows but your documents don't.** If it answers, it is not grounded.
19. **RAG reduces hallucination, it does not eliminate it.** The real win is that errors become citable and therefore detectable.
20. **Chunking is the first thing to check when search quality is bad** — before models, before databases.

---

## 5. Interview Preparation Notes

**Q1. Walk me through building semantic search over a company knowledge base.**
Chunk documents to about 200 words so nothing is silently truncated. Embed each chunk with a model chosen for the language and domain — MiniLM is a sensible default for English. Store vectors with the document text and metadata in a vector database. At query time, embed the query with the *same* model, retrieve the top k, optionally rerank, and pass the results to an LLM instructed to answer only from that context. Before any of that, build an evaluation set of real queries so every subsequent decision is measured.

**Q2. How do you prove semantic search is better than what the client already has?**
A bake-off on their content. Take 30 to 50 real queries from their search logs, mark which documents should be returned, and run their existing keyword search and the semantic version through the identical scoring function. Report hit@3 or recall@5 for both. That is a number a business owner can act on, unlike "it feels smarter".

**Q3. What is the difference between ChromaDB and FAISS?**
Category, not degree. FAISS is a library: an in-memory index that returns row numbers and distances, with no text, metadata, filtering, or persistence unless you build them. Chroma is an embedded database: documents, metadata, filters and durability included, with a vector index inside. Prototypes and small production go to Chroma; large-scale systems where you already own a document store go to FAISS.

**Q4. Explain the tradeoff you tuned with `nprobe`.**
`nprobe` sets how many IVF clusters are searched. Low values skip most of the corpus, so queries are dramatically faster and recall falls — at `nlist=1024, nprobe=1` we measured roughly 99× speedup at 49% recall, which is a broken search engine. At `nprobe=5` recall returned to near perfect while remaining very fast. The general principle: set a recall target first, then tune for latency underneath it.

**Q5. Why hybrid search?**
Because the two methods fail in different places. Semantic search cannot reliably retrieve an exact identifier — to an embedding model, all reference numbers look alike. Keyword search cannot bridge vocabulary. RRF merges them on rank rather than score, since BM25 is unbounded and cosine is 0-to-1, so adding the raw scores lets one method dominate arbitrarily.

**Q6. How do you stop a RAG system from hallucinating?**
Layered. Instruct the model to answer only from the provided context and to state plainly when it cannot. Require citations so any claim is traceable to a document. Test with questions the model knows from training but the corpus does not contain — if it answers, it is not grounded. Log and evaluate that refusal behaviour continuously. And be precise in how you describe the result: this makes errors detectable and traceable; it does not make them impossible.

**Q7. Your search worked in the demo and is poor in production. Where do you look first?**
Chunking. Demo documents are short, production documents are long, and embedding models truncate silently — MiniLM at 256 tokens — so most of each document was never embedded. That single issue explains more production retrieval failures than model choice does. After that: is the evaluation set representative, is the domain vocabulary out of distribution for the model, and are filters being applied correctly.

**Q8 (FDE). The client asks why they should not just paste everything into Claude.**
Capacity and cost. Their 4,000 articles will not fit in a context window, and even the portion that would fit costs orders of magnitude more per query than an embedding lookup. Retrieval narrows 4,000 documents to 3 for a fraction of a cent in milliseconds, and Claude then does the part that genuinely needs intelligence. I would demo both against their real content with the token counts on screen — that conversation ends quickly.

---

## 6. Assignment

**Beginner.** Add 5 new documents about a topic the knowledge base does not cover (say, maternity benefits). Write 2 customer-language queries for them and confirm semantic search finds them while BM25 does not.

**Intermediate.** Extend `evaluate()` to also compute **MRR** (mean reciprocal rank — the average of `1/rank` of the first correct result). Re-score all four methods. Explain a case where MRR and hit@3 disagree, and which one you would report to Priya.

**Advanced.** Take one 3,000-word document. Index it (a) whole and (b) chunked into ~200-word pieces with 20-word overlap. Write 5 queries targeting content in the last third of the document. Measure hit@3 for both. Quantify the damage silent truncation does.

**Project — ship the Kaveri Support Copilot.** Deliver a notebook that contains: a persistent ChromaDB collection with metadata; hybrid retrieval; the Claude copilot with citations; an evaluation table comparing keyword, semantic and hybrid on at least 15 queries; and a documented grounding test proving the copilot refuses out-of-scope questions. Write a one-paragraph recommendation to Priya containing an actual number.

---

## 7. Assessment

### Part A — Multiple choice (10)

1. In Chroma's query result, `results["documents"]` is:
   (a) a string  (b) a list of strings  (c) a list of lists  (d) a dict

2. Chroma's default embedding function is:
   (a) OpenAI text-embedding-3-small  (b) ONNX all-MiniLM-L6-v2  (c) all-mpnet-base-v2  (d) Voyage AI

3. In a cosine-space collection, a `distance` of 0.15 means:
   (a) a weak match  (b) a strong match  (c) 15% relevant  (d) an error

4. `faiss.IndexFlatIP` returns scores where:
   (a) lower is better  (b) higher is better  (c) always positive  (d) always 0 to 1

5. `nprobe` controls:
   (a) result count  (b) how many IVF clusters are searched  (c) vector dimensions  (d) thread count

6. RRF fuses rankings using:
   (a) the sum of raw scores  (b) `1/(60+rank)` summed  (c) the maximum score  (d) cosine of the scores

7. You change the embedding model on an existing collection. You must:
   (a) nothing  (b) restart the client  (c) create a new collection and re-embed everything  (d) change `n_results`

8. Applying a tenant filter to results *after* retrieval is bad mainly because:
   (a) it is slower  (b) another tenant's data entered your process  (c) Chroma forbids it  (d) it breaks cosine

9. A vector search with no relevant documents in the index returns:
   (a) an empty list  (b) an error  (c) its nearest neighbours anyway  (d) None

10. The single most common cause of poor RAG quality in production is:
   (a) the wrong vector database  (b) unchunked long documents  (c) too small `n_results`  (d) temperature

### Part B — Short answer (5)

11. Why must both methods be scored with the identical `evaluate()` function?
12. Why is a "1000× faster" vector search benchmark meaningless on its own?
13. Why does adding BM25 and cosine scores together fail, and what does RRF do instead?
14. Describe the test that proves a RAG system is grounded, and why it must use a question the model already knows.
15. MPNet tied with MiniLM on your 12 queries. What do you conclude, and what do you *not* conclude?

### Part C — Scenario (3)

16. Priya reports that search is excellent for policy questions but useless when customers paste a claim reference number. Diagnose and fix, and say what you would measure to prove the fix worked.

17. You move from 50 documents to 2 million. Which parts of today's notebook survive unchanged, which must be replaced, and what new failure modes appear?

18. The copilot told a customer their dental treatment was covered. It is not. Walk through your investigation, in order, and name the guard you would add at each layer.

---

## 8. Answer Key

**Part A:** 1-(c) · 2-(b) · 3-(b) · 4-(b) · 5-(b) · 6-(b) · 7-(c) · 8-(b) · 9-(c) · 10-(b)

**Part B**

11. Because a comparison is only valid if exactly one variable changes. Same queries, same k, same relevance judgements, same scoring code. Tune the metric separately per method and you are measuring your tuning, not the methods.

12. Because approximate search buys speed by skipping vectors, so speed and recall move in opposite directions. `nprobe=1` is always fast and can be at 49% recall — a search engine that misses half the correct answers. Without a stated recall, the speed number describes nothing.

13. BM25 scores are unbounded (0 to 20+) and cosine similarities sit in 0 to 1, so a raw sum lets BM25 dominate for reasons unrelated to relevance, and any fixed normalisation is corpus-specific and brittle. RRF discards the scores and uses only ranks: `1/(60+rank)` summed across methods. Nothing needs normalising, there is no weight to tune, and documents ranked well by both methods rise to the top.

14. Ask a question the LLM certainly knows from training but which is absent from your documents — "what is the capital of France?". A grounded system refuses; an ungrounded one answers "Paris". It must be a question the model knows, because a question nobody knows produces a refusal for the wrong reason and proves nothing. The test isolates *where the answer came from*.

15. **Conclude:** on this evaluation set the extra cost of MPNet is not justified, so ship MiniLM and revisit when the evaluation set grows. **Do not conclude:** that MPNet is not a better model — it measurably is on large public benchmarks. Twelve easy English queries cannot resolve a small quality difference. The finding is about the *measurement's* resolution, not the models.

**Part C**

16. **Diagnosis:** reference numbers are semantically indistinguishable from each other — an embedding model has no notion that these particular digits matter, so it returns other reference-bearing documents. **Fix:** hybrid retrieval with RRF, so BM25 (where a rare token scores very high) contributes its ranking. Optionally route obviously ID-shaped queries (a regex) straight to exact lookup. **Measure:** extend the evaluation set with 10 reference-number queries and report hit@1 for semantic, keyword and hybrid separately — averaging them into the main scorecard would hide exactly the failure being fixed.

17. **Survives:** the concepts and the interfaces — chunk, embed, store, retrieve, fuse, generate; the evaluation harness; the Claude layer, which does not care about corpus size. **Must be replaced:** in-memory Chroma (2M documents needs a server-backed store such as Qdrant, Milvus or pgvector, or FAISS with your own document store); exact search must become IVF or HNSW; embedding becomes a batch pipeline rather than one `add()` call; BM25 in pure Python must become a real search engine. **New failure modes:** ANN recall loss that no longer shows up in a 12-query test; index build time and memory (2M × 384 × 4 bytes ≈ 3 GB before overhead); staleness as documents change; noisier retrieval because near-duplicate documents multiply; and tenant isolation becoming a correctness requirement rather than a nicety.

18. **In order.** (1) **Was the right document retrieved?** Log the retrieved ids and check whether the exclusion document was among them — if not, it is a retrieval failure and the likely cause is chunking or a missing document. (2) **Did retrieval return a contradiction?** Cosine similarity ranks "covered" and "not covered" as near-identical, so both may be present — this is the most likely cause. (3) **Did the prompt permit the model to choose?** Add an explicit instruction to flag contradictions rather than resolve them silently. (4) **Was the source document itself wrong or stale?** **Guards:** a cross-encoder reranker for negation handling; citations shown in the UI so a human can spot the mismatch; an automated grounding evaluation covering known exclusions, run in CI; and a content-freshness pipeline that re-embeds on edit. **And the process guard:** a customer-facing coverage answer should carry a confidence threshold and route to a human below it, because in insurance a wrong "yes" is a regulatory problem, not just a bad answer.

---

## 9. Sources

- **ChromaDB documentation** — [docs.trychroma.com](https://docs.trychroma.com/)
- **FAISS repository and index guidance** — [github.com/facebookresearch/faiss](https://github.com/facebookresearch/faiss)
- **FAISS announcement, Meta Engineering (2017)** — [engineering.fb.com](https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/)
- **MiniLM model card** (384-d, 22.7M params, 256-token truncation) — [huggingface.co/sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- **MPNet model card** (768-d, 384-token truncation) — [huggingface.co/sentence-transformers/all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2)
- **Claude Messages API** — [platform.claude.com/docs](https://platform.claude.com/docs/en/build-with-claude/working-with-messages)
- **Anthropic on embeddings** (no first-party embedding model; Voyage AI recommended) — [platform.claude.com/docs/en/build-with-claude/embeddings](https://platform.claude.com/docs/en/build-with-claude/embeddings)
- **Reciprocal Rank Fusion** — Cormack, Clarke & Buettcher, SIGIR 2009

---

### 🎓 You finished Day 10

You did not just learn semantic search. **You measured it, broke it, and fixed it.** That is the difference between someone who has read about retrieval and someone who can be trusted to build it.

Go and explain it to someone. If you can walk a non-technical person through why "my hospital bill was not paid" now finds "cashless authorisation declined", you own this topic.